# 12 — Choosing the serving model, and calibrating it

Notebook 11 showed that direct 3-class training reaches only **0.6065** macro-F1
and that the whole gap is the neutral class (F1 0.376). Reading the 3-star
reviews explained why — they are **mixed**, not neutral:

> *"Clean comfy beds Receptionists clueless when asked about the area.
> Rooms are very small."*  — 3 stars

Strong praise and strong complaint in one text. There is no neutral *language*
to learn; 3 stars is a rating bucket, not a linguistic category.

### A hypothesis, tested and rejected

That suggested a fix: train **binary** on the clear-cut reviews only, and let an
abstention band stand in for "mixed". A binary model that never sees a 3-star
review does place them at the centre — mean p = 0.495 — so the idea was sound
in principle.

It still lost. The comparison is run below and kept in the notebook rather than
deleted, because the negative result is the useful part: **an abstention band
enriched for ambiguity is not the same thing as a classifier that has learned
it.** The band catches 3-star reviews at only 1.37x their base rate, while the
3-class model, for all its weakness, learns something real about them.

The band does earn its place for a *different* job — raising binary precision
from 0.8687 to 0.9270 on the reviews it keeps — and that is recorded at the end
as an optional high-precision mode.

**Served model: direct 3-class.**

In [1]:
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
from src.normalize import normalize, NORMALIZE_VERSION
OUT = REPO / "data_en"
LABELS = ["negative", "neutral", "positive"]

from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib
from src.predict import apply_temperature

tr = pd.read_csv(OUT / "train.csv")
calib = pd.read_csv(OUT / "calib.csv")
hold = pd.read_csv(OUT / "hold.csv")
print(f"train {len(tr):,} | calib {len(calib):,} | hold {len(hold):,}")

train 3,523 | calib 587 | hold 587


In [2]:
t0 = time.perf_counter()
model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    LogisticRegression(max_iter=3000, class_weight="balanced"),
).fit(tr.text.fillna(""), tr.label)
print(f"trained in {time.perf_counter()-t0:.1f}s   classes: {list(model.classes_)}")

CLS = list(model.classes_)
ORDER = [CLS.index(l) for l in LABELS]      # reorder to negative/neutral/positive

def logits(df):
    return model.decision_function(df.text.fillna(""))[:, ORDER]

L_cal, L_hold = logits(calib), logits(hold)
y_cal, y_hold = calib.label.values, hold.label.values
idx_cal = np.array([LABELS.index(l) for l in y_cal])

trained in 0.8s   classes: ['negative', 'neutral', 'positive']


## Temperature

Three-class now, so the scaling is fitted on the full 3-way negative
log-likelihood. As always it cannot move the argmax — only the probabilities,
which the aggregate correction and any confidence display depend on.

In [3]:
def nll(T):
    p = apply_temperature(L_cal, T)
    return -np.mean(np.log(np.clip(p[np.arange(len(idx_cal)), idx_cal], 1e-12, None)))

def ece(probs, idx, n_bins=12):
    conf, pred = probs.max(1), probs.argmax(1)
    acc, bins, e = (pred == idx).astype(float), np.linspace(0, 1, n_bins+1), 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() > 5:
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

grid = np.exp(np.linspace(np.log(0.2), np.log(8.0), 500))
T = float(grid[np.argmin([nll(t) for t in grid])])
before, after = apply_temperature(L_cal, 1.0), apply_temperature(L_cal, T)
print(f"T = {T:.4f}")
print(f"ECE {ece(before, idx_cal):.4f} -> {ece(after, idx_cal):.4f}")
print(f"NLL {nll(1.0):.4f} -> {nll(T):.4f}")
print(f"accuracy unchanged: {accuracy_score(idx_cal, before.argmax(1)):.4f} -> "
      f"{accuracy_score(idx_cal, after.argmax(1)):.4f}")

T = 0.7347
ECE 0.0719 -> 0.0380
NLL 0.7807 -> 0.7650
accuracy unchanged: 0.6440 -> 0.6440


## The comparison that decided it

All three on the same unseen holdout, same labels.

In [4]:
pred3 = np.array(LABELS)[apply_temperature(L_hold, T).argmax(1)]

# the rejected alternative: binary trained on clear reviews + fitted band
trc = tr[tr.label != "neutral"]
mb = make_pipeline(TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True),
                   LogisticRegression(max_iter=3000, class_weight="balanced")
                   ).fit(trc.text.fillna(""), (trc.label == "positive").astype(int))
def pbin(df):
    d = mb.decision_function(df.text.fillna(""))
    return apply_temperature(np.column_stack([np.zeros_like(d), d]), 1.0)[:, 1]
pc, ph_ = pbin(calib), pbin(hold)
best = None
for lo in np.arange(.20, .52, .02):
    for hi in np.arange(.48, .86, .02):
        if hi <= lo: continue
        pr = np.where((pc >= lo) & (pc <= hi), "neutral",
                      np.where(pc > hi, "positive", "negative"))
        f = f1_score(y_cal, pr, average="macro", labels=LABELS)
        if best is None or f > best[0]: best = (f, lo, hi)
_, blo, bhi = best
pred_band = np.where((ph_ >= blo) & (ph_ <= bhi), "neutral",
                     np.where(ph_ > bhi, "positive", "negative"))

rows = [("shipped auto-sentiment", hold.auto.values),
        ("binary + fitted band",   pred_band),
        ("direct 3-class",         pred3)]
print(f"{'approach':<26} {'acc':>7} {'macroF1':>8}   per-class F1")
print("-" * 70)
for n, p in rows:
    per = f1_score(y_hold, p, average=None, labels=LABELS)
    print(f"{n:<26} {accuracy_score(y_hold, p):>7.4f} "
          f"{f1_score(y_hold, p, average='macro', labels=LABELS):>8.4f}   " +
          "  ".join(f"{l[:3]}={v:.3f}" for l, v in zip(LABELS, per)))
print()
print(classification_report(y_hold, pred3, labels=LABELS, digits=4))

approach                       acc  macroF1   per-class F1
----------------------------------------------------------------------
shipped auto-sentiment      0.6167   0.5336   neg=0.567  neu=0.251  pos=0.783
binary + fitted band        0.6337   0.5875   neg=0.659  neu=0.331  pos=0.773
direct 3-class              0.6746   0.6258   neg=0.658  neu=0.416  pos=0.803

              precision    recall  f1-score   support

    negative     0.6272    0.6928    0.6584       153
     neutral     0.4029    0.4308    0.4164       130
    positive     0.8387    0.7697    0.8027       304

    accuracy                         0.6746       587
   macro avg     0.6229    0.6311    0.6258       587
weighted avg     0.6871    0.6746    0.6795       587



## The band's real job: a high-precision mode

Rejected for 3-way classification, it still does something valuable. Abstaining
on low-confidence predictions trades coverage for accuracy, which is the right
trade when the output drives an action on an individual review rather than a
population estimate.

In [5]:
P_hold = apply_temperature(L_hold, T)
conf = P_hold.max(1)
print(f"{'min confidence':>15} {'kept':>12} {'accuracy':>10} {'macroF1':>9}")
for thr in (0.0, 0.40, 0.50, 0.60, 0.70):
    m = conf >= thr
    if m.sum() < 30: continue
    print(f"{thr:>15.2f} {m.sum():>6}/{len(m):<5} {accuracy_score(y_hold[m], pred3[m]):>10.4f} "
          f"{f1_score(y_hold[m], pred3[m], average='macro', labels=LABELS):>9.4f}")
print()
print("Reported as an option, not applied by default: the aggregate estimator")
print("in notebook 13 needs the full population, and dropping the uncertain")
print("cases would bias it toward whatever the model finds easy.")

 min confidence         kept   accuracy   macroF1
           0.00    587/587       0.6746    0.6258
           0.40    563/587       0.6892    0.6379
           0.50    444/587       0.7252    0.6583
           0.60    325/587       0.7938    0.6893
           0.70    219/587       0.8721    0.6981

Reported as an option, not applied by default: the aggregate estimator
in notebook 13 needs the full population, and dropping the uncertain
cases would bias it toward whatever the model finds easy.


In [6]:
mp = OUT / "model_en.joblib"
joblib.dump(model, mp, compress=3)
cfg = {"model_repo": str(mp), "revision": "en-tfidf-3class-v1", "backend": "sklearn",
       "labels": LABELS, "class_order": CLS, "max_length": 0,
       "normalize_version": NORMALIZE_VERSION, "temperature": round(T, 4),
       "confusion_matrix": None,
       "rejected_alternative": {"name": "binary + fitted band",
                                "holdout_macro_f1": round(float(
                                    f1_score(y_hold, pred_band, average='macro',
                                             labels=LABELS)), 4)}}
(REPO / "inference_config_en.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")
json.dump({"temperature": round(T, 4), "ece_before": round(ece(before, idx_cal), 4),
           "ece_after": round(ece(after, idx_cal), 4),
           "holdout": {n: {"accuracy": round(float(accuracy_score(y_hold, p)), 4),
                           "macro_f1": round(float(f1_score(y_hold, p, average="macro",
                                                            labels=LABELS)), 4)}
                       for n, p in rows},
           "size_mb": round(mp.stat().st_size/1e6, 2)},
          open(OUT / "results_calibration.json", "w"), indent=2)
print(f"saved {mp.name} ({mp.stat().st_size/1e6:.2f} MB)")

saved model_en.joblib (0.58 MB)
